In [ ]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

### import relevent classes and function

In [ ]:
from langchain.graph import START , END , StateGraph
from typing_extensions import TypedDict
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage , BaseMessage
from langchain_core.runnable import Runnable
from collections.abc import Sequence

# get famimliar with add_messages function

In [ ]:
my_list = add_messages([HumanMessage("hi! i'm Oscar."),
                        AIMessage("Hey, Oscar. How can i assist you?")],
                        [HumanMessage("Could you summarize today's news?")])

In [ ]:
my_list

# define the state


In [ ]:
class State(TypedDict):
    messages: Sequences[BaseMessage]

# Define the nodes

In [ ]:
chat = ChatOpenAI(model = 'gpt-4o',
                 seed = 365,
                 temperature = 0,
                 max_completion_tokens = 100)

In [ ]:
def ask_questions(state: State) -> State:
    print(f'\n------> ENTERRING ask_question:')
    print("what is your question?")
    return State(messages = [HumanMessage(input())])

In [ ]:
ask_question(State(messages = []))

In [ ]:
def chatbot(state: state) -> State:
    response = chat.invoke(state["messages"])
    response.pretty_print()

    return State(messages = [response])

In [ ]:
def ask_another_questions(state: State) -> State:
    print(f'\n------> ENTERING ask_another_question:')
    print("Would you like to ask one more question (yes/no)?")
    return State(messages = [HumanMessage(input())])

In [ ]:
ask_another_question(State(messages = []))

## defining the routing function

In [ ]:
def routing_function(state : State) -> str:

    if state['messages'].content == 'yes':
        return "ask_question"
    else:
        return "__end__"

# define the graph

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("ask_question" , ask_question)
graph.add_node("chatbot" , chatbot)
graph.add_node("ask_another_question" , ask_another_question)

graph.add_edge("START" , "ask_question")
graph.add_edge("ask_question" ,"chatbot")
graph.add_edge("chatbot" , "ask_another_question")
graph.add_conditional_edges(source = 'ask_another_question' , path = routing_function, path_map = {"True":"ask_question" ,'false': "__end__"})


In [ ]:
graph_compiled = graph.compile()

In [ ]:
graph_compiled

## test the graph